# Pattern 4: External Authorization, Agent-Side (OPA)

The MCP server checks OPA before making each tool call. OPA evaluates role-based rules: admins can do anything, managers can read and approve, employees can only read. If OPA denies, the tool call never reaches the service.

The service still only receives the API key.

**What the service sees:** API key only. Same as patterns 1 and 3.

**Weakness:** The MCP server is the sole enforcement point. A compromised MCP server can skip the OPA check entirely. The service cannot verify that authorization occurred.

In [1]:
from framework.runner import PatternRunner
runner = PatternRunner("p04_external_authz_agent")

## The auth code

This is the lesson. Read both files to understand what happens at each boundary.

In [2]:
runner.show_auth_code()

╭──────────────────────────────── p04_external_authz_agent/mcp_auth.py ────────────────────────────────╮
│    1 """Pattern 4: External Authorization, agent-side (OPA).                                         │
│    2                                                                                                 │
│    3 The MCP server checks OPA before making each tool call. OPA evaluates                           │
│    4 role-based rules (agent_side.rego): admins can do anything, managers can                        │
│    5 read and approve, employees can only read. If OPA denies, the tool call                         │
│    6 never reaches the service.                                                                      │
│    7                                                                                                 │
│    8 The service still only receives the API key -- it has no user identity                          │
│    9 and no knowledge that an OPA check occurred.                                                    │
│   10                                                                                                 │
│   11 This is the pattern's weakness: the agent/MCP server is the sole                                │
│   12 enforcement point. A compromised MCP server can skip the OPA check                              │
│   13 entirely. The service cannot verify that authorization happened.                                │
│   14 """                                                                                             │
│   15                                                                                                 │
│   16 import httpx                                                                                    │
│   17                                                                                                 │
│   18 from framework.auth_helpers import decode_jwt                                                   │
│   19 from framework.config import OPA_URL, SHARED_SERVICE_API_KEY                                    │
│   20 from framework.mcp.auth import AuthHandler, AuthorizationDenied                                 │
│   21                                                                                                 │
│   22                                                                                                 │
│   23 class AgentSideOPAHandler(AuthHandler):                                                         │
│   24     async def before_tool_call(self, user_context, tool_name):                                  │
│   25         jwt = user_context.get("jwt")                                                           │
│   26         if not jwt:                                                                             │
│   27             raise AuthorizationDenied("no JWT provided")                                        │
│   28                                                                                                 │
│   29         claims = decode_jwt(jwt)                                                                │
│   30         opa_input = {                                                                           │
│   31             "input": {                                                                          │
│   32                 "user": {                                                                       │
│   33                     "role": claims.get("role"),                                                 │
│   34                     "department": claims.get("department"),                                     │
│   35                     "reports_to": claims.get("reports_to"),                                     │
│   36                 },                                                                              │
│   37                 "tool": tool_name,                                                              │
│   38    

╭───────────────────── p04_external_authz_agent/service_auth.py ──────────────────────╮
│    1 """Pattern 4: External Authorization, agent-side (service side).               │
│    2                                                                                │
│    3 The service only sees the API key. The OPA check happened at the MCP           │
│    4 server before the request was sent. The service has no way to verify           │
│    5 that any authorization occurred. Intentionally identical to pattern 1.         │
│    6 """                                                                            │
│    7                                                                                │
│    8 from fastapi import Request                                                    │
│    9 from framework.services.identity import Identity                               │
│   10                                                                                │
│   11 EXPECTED_API_KEY = "dev-shared-api-key"                                        │
│   12                                                                                │
│   13                                                                                │
│   14 async def get_expense_identity(request: Request) -> Identity:                  │
│   15     api_key = request.headers.get("x-api-key")                                 │
│   16     if not api_key:                                                            │
│   17         return Identity(method="none", detail="no auth provided")              │
│   18     if api_key == EXPECTED_API_KEY:                                            │
│   19         return Identity(                                                       │
│   20             method="api_key",                                                  │
│   21             detail="shared service credential, no user identity",              │
│   22         )                                                                      │
│   23     return Identity(                                                           │
│   24         method="none",                                                         │
│   25         detail=f"X-API-Key did not match (received prefix: {api_key[:8]}...)", │
│   26     )                                                                          │
│   27                                                                                │
│   28                                                                                │
│   29 get_document_identity = get_expense_identity                                   │
│   30                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────╯

before_tool_call queries OPA with the user's role, tool name, and action. If OPA denies, AuthorizationDenied is raised and the request never reaches the service.

But look at service_auth.py: identical to pattern 1. The service has no user identity and no knowledge that an OPA check happened.

### What's in the OPA policy?

The agent-side policy (`infra/opa/agent_side.rego`) is a role-based check written in [Rego](https://www.openpolicyagent.org/docs/latest/policy-language/), OPA's declarative policy language:

```rego
# Default deny -- every request is denied unless a rule explicitly allows it.
default allow := false

# Admins can call any tool.
allow if { input.user.role == "admin" }

# Managers can call read tools and approve_expense.
allow if {
    input.user.role == "manager"
    input.tool in {"get_expenses", "search_documents", "approve_expense"}
}

# Employees can call read tools only -- they cannot approve expenses.
allow if {
    input.user.role == "employee"
    input.tool in {"get_expenses", "search_documents"}
}
```

The input comes from the MCP server: `{user: {role, department, reports_to}, tool, action}`. OPA evaluates the rules and returns `{allow: true/false, reason: "..."}`. The full policy is in `infra/opa/agent_side.rego`.

In [3]:
await runner.start()

[04/14/26 06:25:25] INFO     StreamableHTTP session manager started                  ]8;id=536746;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=608042;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     StreamableHTTP session manager started                  ]8;id=472083;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=279113;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:49904/mcp "HTTP/1.1 200 OK"        ]8;id=249634;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=579390;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=373148;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=643488;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

                    INFO     Terminating session: None                                       ]8;id=296673;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=478960;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

                    INFO     HTTP Request: POST http://127.0.0.1:49905/mcp "HTTP/1.1 200 OK"        ]8;id=39286;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=791993;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:49905/mcp "HTTP/1.1 200 OK"        ]8;id=656554;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=92077;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:49905/mcp "HTTP/1.1 200 OK"        ]8;id=522917;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=563685;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=669293;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=453279;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=752493;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=985902;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

## Run scenarios

Let's see this pattern in action with different users.

In [4]:
await runner.run_as("alice", "What are my expenses?")

                    INFO     HTTP Request: POST                                                     ]8;id=269977;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=980419;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭───────────────────────────────╮
│ [alice] What are my expenses? │
╰───────────────────────────────╯

[04/14/26 06:25:26] INFO     Terminating session: None                                       ]8;id=865445;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=187268;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

                    INFO     Processing request of type ListToolsRequest                              ]8;id=353874;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=872435;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

                    INFO     Terminating session: None                                       ]8;id=369144;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=808170;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

[04/14/26 06:25:31] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14840;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=407311;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:25:48] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=617912;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=227259;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args ┃ status ┃ result                                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {}   │      - │ {'type': 'text', 'text': '{"identity_method": "api_key",                   │
│     │              │      │        │ "identity_detail": "shared service credential, no user identity", "count": │
│     │              │      │        │ 8, "expenses": [{"id": 1, "user_id": "alice", "department": "engineering", │
│     │              │      │        │ "amount": 42.5, "category": "software", "description": "JetBrains AI       │
│     │              │      │        │ assistant subscription",                                                   │
└─────┴──────────────┴──────┴────────┴────────────────────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are the expenses visible to your account (note: this session uses a shared service credential, so you may  │
│ see records for multiple users). If you want to filter to your own user, tell me your user id or the            │
│ department.                                                                                                     │
│                                                                                                                 │
│ Summary                                                                                                         │
│ - Total expenses: $3,344.50                                                                                     │
│ - Approved: $1,894.50                                                                                           │
│ - Pending: $1,450.00                                                                                            │
│                                                                                                                 │
│ Expense details                                                                                                 │
│ - ID 1: $42.50 | Engineering | Software | JetBrains AI assistant subscription | alice | approved                │
│ - ID 2: $156.00 | Engineering | Travel | Train ticket to client offsite | alice | approved                      │
│ - ID 3: $89.00 | Engineering | Books | Designing Data-Intensive Applications | alice | approved                 │
│ - ID 4: $1,450.00 | Engineering | Hardware | External 4K monitor | alice | pending                              │
│ - ID 5: $320.00 | Engineering | Training | OAuth 2.0 deep-dive workshop | bob | approved                        │
│ - ID 6: $67.00 | Engineering | Meals | Team lunch after the migration shipped | bob | approved                  │
│ - ID 7: $980.00 | Platform | Training | KubeCon ticket | dave | approved                                        │
│ - ID 8: $240.00 | Platform | Software | Datadog license seat | dave | approved                                  │
│                                                                                                                 │
│ Would you like me to filter to a specific user or department, or show only your expenses?                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are the expenses visible to your account (note: this session uses a shared service credential, so you may see records for multiple users). If you want to filter to your own user, tell me your user id or the department.\n\nSummary\n- Total expenses: $3,344.50\n- Approved: $1,894.50\n- Pending: $1,450.00\n\nExpense details\n- ID 1: $42.50 | Engineering | Software | JetBrains AI assistant subscription | alice | approved\n- ID 2: $156.00 | Engineering | Travel | Train ticket to client offsite | alice | approved\n- ID 3: $89.00 | Engineering | Books | Designing Data-Intensive Applications | alice | approved\n- ID 4: $1,450.00 | Engineering | Hardware | External 4K monitor | alice | pending\n- ID 5: $320.00 | Engineering | Training | OAuth 2.0 deep-dive workshop | bob | approved\n- ID 6: $67.00 | Engineering | Meals | Team lunch after the migration shipped | bob | approved\n- ID 7: $980.00 | Platform | Training | KubeCon ticket | dave | approved\n- ID 8: $240.00 | P

In [5]:
# Expected: OPA denies this -- employees cannot approve expenses
await runner.run_as("alice", "Approve expense 4")

                    INFO     HTTP Request: POST                                                     ]8;id=366114;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=113382;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭───────────────────────────╮
│ [alice] Approve expense 4 │
╰───────────────────────────╯

[04/14/26 06:25:50] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=342581;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=231394;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=269602;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=66054;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=272249;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=495766;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8181/v1/data/agentauth/agent_side/decision "HTTP/1.1                 
                             200 OK"                                                                               

...

[04/14/26 06:25:51] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=471652;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=925999;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:26:07] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=286409;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=977679;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool            ┃ args              ┃ status ┃ result                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ approve_expense │ {'expense_id': 4} │      - │ {'type': 'text', 'text': '{"_status": 403, "error": "OPA   │
│     │                 │                   │        │ denied alice calling approve_expense: role not permitted   │
│     │                 │                   │        │ to invoke this tool", "_denied_by": "agent_side_opa"}'}    │
└─────┴─────────────────┴───────────────────┴────────┴────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ I tried to approve expense 4, but you don’t have permission to approve expenses in this system. The call        │
│ returned a 403 authorization error.                                                                             │
│                                                                                                                 │
│ Would you like me to:                                                                                           │
│ - Escalate it to a manager/admin, or                                                                            │
│ - Check for any expenses you are allowed to approve, if applicable?                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='I tried to approve expense 4, but you don’t have permission to approve expenses in this system. The call returned a 403 authorization error.\n\nWould you like me to:\n- Escalate it to a manager/admin, or\n- Check for any expenses you are allowed to approve, if applicable?', tool_calls=[ToolCallTrace(name='approve_expense', args={'expense_id': 4}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"_status": 403, "error": "OPA denied alice calling approve_expense: role not permitted to invoke this tool", "_denied_by": "agent_side_opa"}\'}', error=None)])

In [6]:
# Expected: OPA allows this -- managers can approve for their reports
await runner.run_as("bob", "Approve expense 4")

                    INFO     HTTP Request: POST                                                     ]8;id=850700;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=843931;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭─────────────────────────╮
│ [bob] Approve expense 4 │
╰─────────────────────────╯

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=120591;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=351888;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=164467;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=823806;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:26:10] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=885406;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=112044;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

...

[04/14/26 06:26:12] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=415964;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=333685;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:26:15] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=999980;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=679155;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool            ┃ args              ┃ status ┃ result                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ approve_expense │ {'expense_id': 4} │      - │ {'type': 'text', 'text': '{"identity_method": "api_key",   │
│     │                 │                   │        │ "approved_by": "<api-key-caller>",                         │
│     │                 │                   │        │ "tool_side_authz_enabled": false, "expense": {"id": 4,     │
│     │                 │                   │        │ "user_id": "alice", "department": "engineering", "amount": │
│     │                 │                   │        │ 1450.0, "category": "hardware", "description": "External   │
│     │                 │                   │        │ 4K monitor", "status": "approved"}, "                      │
└─────┴─────────────────┴───────────────────┴────────┴────────────────────────────────────────────────────────────┘

╭────────────────────────────────── answer ──────────────────────────────────╮
│ Expense 4 approved.                                                        │
│                                                                            │
│ Details: Alice (Engineering) – Hardware – External 4K monitor – $1,450.00. │
╰────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Expense 4 approved.\n\nDetails: Alice (Engineering) – Hardware – External 4K monitor – $1,450.00.', tool_calls=[ToolCallTrace(name='approve_expense', args={'expense_id': 4}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "api_key", "approved_by": "<api-key-caller>", "tool_side_authz_enabled": false, "expense": {"id": 4, "user_id": "alice", "department": "engineering", "amount": 1450.0, "category": "hardware", "description": "External 4K monitor", "status": "approved"}, "', error=None)])

In [7]:
await runner.run_as("dave", "Search for compensation documents")

                    INFO     HTTP Request: POST                                                     ]8;id=636498;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=961134;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────────╮
│ [dave] Search for compensation documents │
╰──────────────────────────────────────────╯

[04/14/26 06:26:18] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=455474;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=851883;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:26:20] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=683073;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=512970;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=975927;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=926842;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

...

[04/14/26 06:26:33] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=874896;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=927715;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:26:42] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=265898;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=885078;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool             ┃ args                            ┃ status ┃ result                                      ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ search_documents │ {'q': 'compensation documents'} │      - │ {'type': 'text', 'text':                    │
│     │                  │                                 │        │ '{"identity_method": "api_key",             │
│     │                  │                                 │        │ "identity_detail": "shared service          │
│     │                  │                                 │        │ credential, no user identity",              │
│     │                  │                                 │        │ "allowed_groups": "all", "count": 0,        │
│     │                  │                                 │        │ "documents": [], "_status": 200}'}          │
│ 2   │ search_documents │ {'q': 'compensation'}           │      - │ {'type': 'text', 'text':                    │
│     │                  │                                 │        │ '{"identity_method": "api_key",             │
│     │                  │                                 │        │ "identity_detail": "shared service          │
│     │                  │                                 │        │ credential, no user identity",              │
│     │                  │                                 │        │ "allowed_groups": "all", "count": 1,        │
│     │                  │                                 │        │ "documents": [{"id": 6, "title":            │
│     │                  │                                 │        │ "Compensation bands H1 2026", "body": "L3   │
│     │                  │                                 │        │ 75-95k, L4 95-130k, L5 130-180k, L6         │
│     │                  │                                 │        │ 180-240k. Confidential.", "ow               │
│ 3   │ search_documents │ {'q': 'salary policy'}          │      - │ {'type': 'text', 'text':                    │
│     │                  │                                 │        │ '{"identity_method": "api_key",             │
│     │                  │                                 │        │ "identity_detail": "shared service          │
│     │                  │                                 │        │ credential, no user identity",              │
│     │                  │                                 │        │ "allowed_groups": "all", "count": 0,        │
│     │                  │                                 │        │ "documents": [], "_status": 200}'}          │
└─────┴──────────────────┴─────────────────────────────────┴────────┴─────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are the compensation documents I found.                                                                    │
│                                                                                                                 │
│ - Document ID: 6                                                                                                │
│   - Title: Compensation bands H1 2026                                                                           │
│   - Snippet: L3 75-95k, L4 95-130k, L5 130-180k, L6 180-240k. Confidential.                                     │
│   - Access: Admin only (owner: dave)                                                                            │
│                                                                                                                 │
│ Note: This document is marked Confidential and restricted to admins. If you need full content, you’ll need      │
│ admin access.                                                                                                   │
│                                                                                                                 │
│ Other search attempt:                                                                                           │
│ - Query "salary policy" returned no results.                                                                    │
│                                                                                                                 │
│ Would you like me to broaden the search to terms like "compensation policy", "remuneration policy", or search   │
│ by a specific department? Or should I try a different query or filter?                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are the compensation documents I found.\n\n- Document ID: 6\n  - Title: Compensation bands H1 2026\n  - Snippet: L3 75-95k, L4 95-130k, L5 130-180k, L6 180-240k. Confidential.\n  - Access: Admin only (owner: dave)\n\nNote: This document is marked Confidential and restricted to admins. If you need full content, you’ll need admin access.\n\nOther search attempt:\n- Query "salary policy" returned no results.\n\nWould you like me to broaden the search to terms like "compensation policy", "remuneration policy", or search by a specific department? Or should I try a different query or filter?', tool_calls=[ToolCallTrace(name='search_documents', args={'q': 'compensation documents'}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "api_key", "identity_detail": "shared service credential, no user identity", "allowed_groups": "all", "count": 0, "documents": [], "_status": 200}\'}', error=None), ToolCallTrace(name='search_documents', args

## What did the service see?

The punchline: what identity did the backend service actually extract?

In [8]:
await runner.show_service_identity()

[04/14/26 06:26:43] INFO     HTTP Request: GET http://127.0.0.1:49902/debug/last-request "HTTP/1.1  ]8;id=526254;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=767453;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭──────── expense-service /debug/last-request ─────────╮
│ method:  api_key                                     │
│ user_id: <none>                                      │
│ detail:  shared service credential, no user identity │
│                                                      │
╰──────────────────────────────────────────────────────╯

                    INFO     HTTP Request: GET http://127.0.0.1:49903/debug/last-request "HTTP/1.1  ]8;id=402084;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=797006;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭──────── document-service /debug/last-request ────────╮
│ method:  api_key                                     │
│ user_id: <none>                                      │
│ detail:  shared service credential, no user identity │
│                                                      │
╰──────────────────────────────────────────────────────╯

Alice's approve attempt was blocked by OPA before the request reached the service. Bob's succeeded because OPA allows managers to approve. But the service still reports method=api_key.

The OPA check is centralized and maintainable, which is better than pattern 3's scattered logic. But it only runs at the MCP layer. The service has no proof that any authorization occurred.

We need the service to verify the user's identity independently.

In [9]:
await runner.stop()

Pattern p04_external_authz_agent stopped.

[04/14/26 06:26:44] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=87193;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=849935;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       